# Custom networks, data and energy distributions

A network is built from plain objects: species, states, rate models and reactions, each reaction with the provenance of its data. This tutorial builds a three-level atom, checks and saves it, and then solves the hydrogen model with a non-Maxwellian electron energy distribution.

In [ ]:
from plasmacoll import CrossSection

import opencr
from opencr.core import (
    ELECTRON_STATE as e,
)
from opencr.core import (
    Constant,
    CrossSectionRate,
    DetailedBalance,
    Reaction,
    ReactionNetwork,
    State,
    species_from_formula,
)
from opencr.data import (
    Provenance,
    Reference,
    check_network_data,
    load_network,
    save_network,
)

## A three-level atom

The cross sections here are made up (a constant above threshold), and the provenance says so.

In [ ]:
atom = species_from_formula("He")
ground = State(atom, "1", 0.0, 1.0, {"n": 1})
meta = State(atom, "2", 19.8, 3.0, {"n": 2})
upper = State(atom, "3", 23.0, 9.0, {"n": 3})
made_up = Provenance(Reference("illustration only"), units="m^2 against eV", licence="MIT")
network = ReactionNetwork(
    [ground, meta, upper],
    [
        Reaction("exc12", (e, ground), (e, meta), "excitation",
                 CrossSectionRate(cross_section=CrossSection.constant(1e-21, 19.8)), made_up),
        Reaction("exc13", (e, ground), (e, upper), "excitation",
                 CrossSectionRate(cross_section=CrossSection.constant(5e-22, 23.0)), made_up),
        Reaction("dex21", (e, meta), (e, ground), "deexcitation", DetailedBalance(forward="exc12")),
        Reaction("dex31", (e, upper), (e, ground), "deexcitation", DetailedBalance(forward="exc13")),
        Reaction("A32", (upper,), (meta,), "spontaneous_emission", Constant(value=1e7), made_up),
    ],
    name="three-level",
    assumptions=["made-up data"],
)
print(network)
print(check_network_data(network))

The network is saved in OpenCR's versioned JSON format and read back:

In [ ]:
import tempfile
from pathlib import Path

path = save_network(network, Path(tempfile.mkdtemp()) / "three_level.json")
again = load_network(path)
print(again, again.reaction("dex21").rate)

Solved with the ground state as the driver:

In [ ]:
model = opencr.Model(again, drivers=["He(1)"])
result = model.solve_steady_state(opencr.Conditions(5.0, 1e18, densities={"He(1)": 1e20}))
print({k: f"{v:.3e}" for k, v in result.populations.items()})

## Non-Maxwellian electrons

An EEDF with fewer fast electrons than a Maxwellian of the same mean energy (a Druyvesteyn distribution) lowers the rates of processes with high thresholds. `Conditions.from_eedf` sets the electron temperature to $2/3\,\langle\varepsilon\rangle$, used by the processes that are not cross sections and by three-body recombination; cross sections are averaged over the distribution itself. A tabulated distribution from a probe, or from plasmacoll's Boltzmann solver (`TabulatedEEDF.from_boltzmann`), works the same way.

In [ ]:
from opencr.kinetics.eedf import Generalized, Maxwellian

hydrogen = opencr.load_model("hydrogen/atomic/basic", n_max=12)
drivers = {"H(n=1)": 1.0, "H+": 1.0}
for mean in (3.0, 6.0, 15.0):
    rates = []
    for eedf in (Maxwellian(2 * mean / 3), Generalized(mean, 2.0)):
        q = hydrogen.effective_rates(opencr.Conditions.from_eedf(eedf, electron_density=1e18, densities=drivers))
        rates.append(q.effective_rate("H(n=1)", "H+"))
    print(f"<eps> = {mean:4.1f} eV: S_CR Maxwellian {rates[0]:.3e}, Druyvesteyn {rates[1]:.3e} m^3/s")